## 10.5. Multihead Attention

### 10.5.1 Model

$$
\textbf{h}_i = f(\textbf{W}_i^{(q)}\textbf{q}, \textbf{W}_i^{(k)}\textbf{k}, \textbf{W}_i^{(v)}\textbf{v}) \in \mathbb{R}^{p_v}
$$

$\textbf{W}_i^{(q)} \in \mathbb{R}^{p_d × d_q}, \textbf{W}_i^{(k)} \in \mathbb{R}^{p_k × d_k}, \textbf{W}_i^{(v)} \in \mathbb{R}^{p_v × d_v}, \textbf{W}_o \in \mathbb{R}^{p_o × h_{p_v}}$


$$
\textbf{W}_o\left[\begin{aligned} \textbf{h}_1 \\ ⋮ \\ \text{h}_h
\end{aligned}\right] \in \mathbb{R}^{p_o}
$$

In [1]:
import math
import torch
from torch import nn
from d2l import torch as d2l

### 10.5.2 Implementation

In [7]:
#@save
class MultiHeadAttention(nn.Module):
  """Multi head attention"""
  def __init__(self, key_size, query_size, value_size, num_hiddens, num_heads, dropout, bias=False, **kwargs):
    super(MultiHeadAttention, self).__init__(**kwargs)
    self.num_heads = num_heads
    self.attention = d2l.DotProductAttention(dropout)
    self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
    self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
    self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
    self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)
    
  def forward(self, queries, keys, values, valid_lens):
    # queries, keys, value shape (resp):
    # (batch_size, query|"k-v" pair num, num_hiddens)
    # the same as valid_lens
    # (batch_size, ) || (batch_size, q_num)
    # after transform, output `queries, keys, values` shape (resp)
    # (batch_size * num_heads, query|"k-v" pair num, num_hiddens/num_heads)
    queries = transpose_qkv(self.W_q(queries), self.num_heads)
    keys = transpose_qkv(self.W_k(keys), self.num_heads)
    values = transpose_qkv(self.W_v(values), self.num_heads)
    
    if valid_lens is not None:
      # at ax0, copy first term (scala or vector) num_heads times
      # and copy the second term the same, e.t.c
      valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)
      
    # output shape: (batch_size*num_heads, q_num, num_hiddens/num_heads)
    output = self.attention(queries, keys, values, valid_lens)
    
    # output_concat shape: (batch_size, q_num, num_hiddens)
    output_concat = transpose_output(output, self.num_heads)
    return self.W_o(output_concat)

In [8]:
#@save
def transpose_qkv(X, num_heads):
  """for MHA parallliz to transform"""
  # input X shape: (batch_size, query | "k-v" pair num, num_hiddens)
  # output X shape: (batch_size, query | "k-v" pair num, num_heads
  # num_hiddens / num_heads)
  X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
  
  # output X.shape: (batch_size, num_heads, q | 'k-v' pair num, num_hiddens / num_heads)
  X = X.permute(0, 2, 1, 3)
  
  # ulti output shape: (batch_size*num_heads, q|'k-v' pair num, num_hiddens/num_heads)
  return X.reshape(-1, X.shape[2], X.shape[3])

#@save
def transpose_output(X, num_heads):
  """inverse transpose_qkv function """
  X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
  X = X.permute(0, 2, 1, 3)
  return X.reshape(X.shape[0], X.shape[1], -1)

In [11]:
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens, num_hiddens, num_heads, 0.5)
attention.eval()

MultiHeadAttention(
  (attention): DotProductAttention(
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (W_q): Linear(in_features=100, out_features=100, bias=False)
  (W_k): Linear(in_features=100, out_features=100, bias=False)
  (W_v): Linear(in_features=100, out_features=100, bias=False)
  (W_o): Linear(in_features=100, out_features=100, bias=False)
)

In [12]:
batch_size, num_queries = 2, 4
num_kvpairs, valid_lens = 6, torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
attention(X, Y, Y, valid_lens).shape

torch.Size([2, 4, 100])